- frame_dir (str): The identifier of the corresponding video. (name of file)
- total_frames (int): The number of frames in this video. (len of 'keypoints')
- img_shape (tuple[int]): The shape of a video frame, a tuple with two elements, in the format of (height, width). Only required for 2D skeletons. (got it)
- original_shape (tuple[int]): Same as img_shape. (got it)
- label (int): The action label. ('overhead press')
- keypoint (np.ndarray, with shape [M x T x V x C]): The keypoint annotation. M: number of persons; T: number of frames (same as total_frames); V: number of keypoints (25 for NTURGB+D 3D skeleton, 17 for CoCo, 18 for OpenPose, etc. ); C: number of dimensions for keypoint coordinates (C=2 for 2D keypoint)
- keypoint_score (np.ndarray, with shape [M x T x V]): The confidence score of keypoints. Only required for 2D skeletons.


# Load JSONs


In [ ]:
# %pip install pandas

import pandas as pd
import json
import pickle
import os

In [ ]:
# Settings
base_dir = '../../data'
# sample_class = 'correct'  # 'knees_error', 'elbows_error'
# sample_class = 'knees_error' #'correct'  'knees_error', 'elbows_error'
sample_class = 'elbows_error'  # 'knees_error', 'elbows_error'

extract_main_person = False

In [ ]:
# Path to the folder with JSON files
json_folder = os.path.join(base_dir, 'ohp_poses', sample_class)

# Dictionary to store all loaded JSON data
all_data = {}

# Loop through all .json files in the folder
for filename in os.listdir(json_folder):
    if filename.endswith('.json'):
        filepath = os.path.join(json_folder, filename)
        with open(filepath, 'r') as file:
            try:
                data = json.load(file)
                key = os.path.splitext(filename)[0]  # filename without .json
                all_data[key] = data
            except json.JSONDecodeError:
                print(f"⚠️ Could not parse {filename}, skipping.")

# Example: print one loaded entry
print(all_data.keys())

In [ ]:
# print('No. people: ',len(all_data.get('62794_6').get('keypoints')[0].keys()))

all_data


In [ ]:
# print('No. frames: ', len(all_data.get('62794_6').get('keypoints')))

In [ ]:
# %pip install opencv-python

# Get the information of the VIDEOS


In [ ]:
import cv2

# Path to the folder containing .mp4 videos
video_folder = os.path.join(base_dir, 'ohp_labeled', sample_class)

# Dictionary to hold video metadata
video_info = {}

# Loop through all files in the folder
for filename in os.listdir(video_folder):
    if filename.lower().endswith('.mp4'):
        video_path = os.path.join(video_folder, filename)
        video_name = os.path.splitext(filename)[0]

        # Open video file
        vid = cv2.VideoCapture(video_path)

        if not vid.isOpened():
            print(f"❌ Failed to open: {filename}")
            continue

        # Get properties
        width = vid.get(cv2.CAP_PROP_FRAME_WIDTH)
        height = vid.get(cv2.CAP_PROP_FRAME_HEIGHT)
        fps = vid.get(cv2.CAP_PROP_FPS)
        frame_count = vid.get(cv2.CAP_PROP_FRAME_COUNT)
        duration = frame_count / fps if fps else 0

        # Store in dictionary
        video_info[video_name] = {
            "width": int(width),
            "height": int(height),
            "fps": round(fps, 2),
            "frame_count": int(frame_count),
            "duration_sec": round(duration, 2)
        }

        vid.release()

# Print or save the results
output_path = os.path.join(video_folder, 'video_properties.json')
with open(output_path, 'w') as f:
    json.dump(video_info, f, indent=2)

print(f"✅ Processed {len(video_info)} videos. Info saved to: {output_path}")

In [ ]:
with open(os.path.join(video_folder, 'video_properties.json'), 'r') as file:
    video_properties = json.load(file)

In [ ]:
video_properties

# Detection of people in the videos


In [ ]:
# this detects the number of people in each video, in the frame keypoints[1] (the second person)

people_lst = []
for i in all_data.keys():
    print('No. people: ',len(all_data.get(i).get('keypoints')[1].keys()))
    people_lst.append([i, len(all_data.get(i).get('keypoints')[1].keys())])

In [ ]:
# videos with no people in the second frame
for i in people_lst:
    if i[1]==0:
        print(i)




In [ ]:
# How many people are in each video at most?

people_lst = []
for i in all_data.keys():
    people_visible = []
    for j in range(len(all_data.get(i).get('keypoints'))):
        if len(all_data.get(i).get('keypoints')[j])>0:
            people_visible.append(len(all_data.get(i).get('keypoints')[j].keys()))
    people_lst.append([i, max(people_visible)])
people_lst


In [ ]:
# Ressume of the max number of people per video
# Conclusion: is worth it to work with the 2 and 3 people videos.

from collections import Counter

# Get only the max number of people per video
max_people_per_video = [item[1] for item in people_lst]

# Count how many times each number appears
summary = Counter(max_people_per_video)

# Print summary sorted by number of people
for num_people in sorted(summary):
    print(f"{summary[num_people]} videos with {num_people} people at most")


In [ ]:
# Get ID_video with max number of people per video, only when is greater than 1

from collections import defaultdict

# Diccionario para agrupar por número de personas máximas por video
videos_by_people_count = defaultdict(list)

# Obtener la cantidad máxima de personas por video
for video_id, video_data in all_data.items():
    max_people = 0
    for frame in video_data['keypoints']:
        people_in_frame = len(frame)
        if people_in_frame > max_people:
            max_people = people_in_frame

    if max_people > 1:
        videos_by_people_count[max_people].append(video_id)

# Mostrar resultados como: id video | N people in screen
for people_count in sorted(videos_by_people_count.keys()):
    for video_id in videos_by_people_count[people_count]:
        print(f"{video_id} | {people_count} people in screen")


In [ ]:
# # Copy the VIDEOS to a new folder to manually check them
# # Conclusion: nothing really crazy happening here, the videos are quite normal. The main person is visible most of the time.


# import os
# import shutil

# # Crear lista de videos con más de una persona
# multi_person_videos = []

# for video_id, video_data in all_data.items():
#     max_people = max(len(frame) for frame in video_data['keypoints'])
#     if max_people > 1:
#         multi_person_videos.append(video_id)

# # Definir carpetas
# destination_base = os.path.join(base_dir, "videos with multiple people")
# destination_class_folder = os.path.join(destination_base, sample_class)
# source_video_folder = os.path.join(base_dir, 'ohp_labeled', sample_class)

# # Crear carpetas si no existen
# os.makedirs(destination_class_folder, exist_ok=True)

# # Copiar los archivos de video
# for video_id in multi_person_videos:
#     source_file = os.path.join(source_video_folder, f"{video_id}.mp4")
#     destination_file = os.path.join(destination_class_folder, f"{video_id}.mp4")

#     if os.path.exists(source_file):
#         shutil.copy2(source_file, destination_file)
#         print(f"✅ Copied: {video_id}.mp4")
#     else:
#         print(f"⚠️ Video not found: {video_id}.mp4")

# print(f"\n🎯 Completed copying {len(multi_person_videos)} videos to {destination_class_folder}")


In [ ]:
for i in people_lst:
    if i[1]>3:
        print(i)

In [ ]:
# # Define boxes for each person in the frame

# def get_bounding_box(keypoints, threshold=0.0):
#     """
#     keypoints: list of (x, y, confidence) or (x, y)
#     """
#     valid_points = []
#     for kp in keypoints:
#         if len(kp) == 3:
#             x, y, conf = kp
#             if conf >= threshold:
#                 valid_points.append((x, y))
#         elif len(kp) == 2:
#             x, y = kp
#             valid_points.append((x, y))

#     if not valid_points:
#         return None

#     xs, ys = zip(*valid_points)
#     return min(xs), min(ys), max(xs), max(ys)

In [ ]:
# def bbox_area(bbox):
#     x_min, y_min, x_max, y_max = bbox
#     return (x_max - x_min) * (y_max - y_min)

In [ ]:
#all_data.get('80557_5').get('keypoints')

In [ ]:
#video_properties.get('80557_5')

In [ ]:
len(all_data.keys())

# Nacho: Track same people through the different frames


In [ ]:
import os
from collections import defaultdict

# Configuration parameters
frame_gap_threshold = 3  # Number of frames that counts as a significant disappearance

# Output folder for the corrected JSONs (not yet used but prepared)
json_output_folder = os.path.join(base_dir, 'ohp_poses_corrected', sample_class)
os.makedirs(json_output_folder, exist_ok=True)

# Dictionary to hold videos that may require ID tracking
candidates_for_tracking = {}

# Only process videos where more than 1 person appears at some point
for video_id, video_data in all_data.items():
    keypoints = video_data['keypoints']
    person_frames = defaultdict(list)  # {person_id: [frame indices where person appears]}

    # Collect which frames each person_id appears in
    for frame_idx, frame_data in enumerate(keypoints):
        for person_id in frame_data.keys():
            person_frames[person_id].append(frame_idx)

    # Look for person IDs that disappear and reappear (i.e., have big gaps)
    fragmented_ids = []
    for pid, frames in person_frames.items():
        if len(frames) < 2:
            continue  # skip IDs that appear only once
        # Calculate the frame-to-frame gaps
        gaps = [b - a for a, b in zip(frames[:-1], frames[1:])]
        max_gap = max(gaps) if gaps else 0
        if max_gap >= frame_gap_threshold:
            fragmented_ids.append((pid, max_gap))

    # If we found any ID with gaps, mark this video for tracking
    if fragmented_ids:
        candidates_for_tracking[video_id] = {
            "fragmented_ids": fragmented_ids,
            "person_frames": dict(person_frames)
        }

print(f"🎯 Detected {len(candidates_for_tracking)} videos with potential ID fragmentation.\n")

# Example output: show first 5 videos with issues
for vid, data in list(candidates_for_tracking.items())[:5]:
    print(f"📹 Video: {vid}")
    for pid, gap in data["fragmented_ids"]:
        print(f"   ⚠️ Person ID {pid} has a gap of {gap} frames")
    print("")


# Stamatia: take only the "main person" from the JSONs


In [ ]:
# # Dont use this if you want all the people in the video.
# # This bbg extracts "the main person" from each video based on the largest bounding box area of keypoints.
# # This "main person" is defined as the person with the largest bounding box in the **first frame** of each video.
# # Should redefine that 


# main_person = None
# counter = 0
# main_person_keypoints = {}

# if not extract_main_person:
#     exit()

# for video in all_data.keys():
#     main_person = None
#     largest_area = 0
#     for frame in all_data.get(video).get('keypoints'):
#         if len(frame.keys())>0:
#             for (person, person_keypoints) in frame.items():  # each is a list of keypoints
#                 bbox = get_bounding_box(person_keypoints, threshold=0.2)  # optional threshold
#                 if bbox:
#                     area = bbox_area(bbox)
#                     if area > largest_area:
#                         largest_area = area
#                         main_person = {
#                             "bbox": bbox,
#                             "keypoints": person_keypoints,
#                             "area": area,
#                             "person_id": person
#                         }
#             counter+=1
#             if main_person:
#                 print("Main person bounding box:", main_person["bbox"], video, counter)
#                 print(main_person['area'])
#                 print(main_person['person_id'])
#             break
#     main_persons_frames = []
#     for frame in all_data.get(video).get('keypoints'):
#         if frame.get(main_person['person_id']):
#             main_persons_frames.append(frame.get(main_person['person_id'])[:17])
        
#     main_person_keypoints[video] = {main_person['person_id']: main_persons_frames}


# From JSON to PKL


In [ ]:
import itertools

all_keypoints= {}
for video in all_data.keys():
    persons_frames = {}
    all_keypoints[video] = []
     
    people_lst = list([list(all_data.get(video).get('keypoints')[i].keys()) for i in range(len(all_data.get(video).get('keypoints')))])
    people_set = list(set(itertools.chain.from_iterable(people_lst)))
    for person in people_set:
            persons_frames[person] = []
    
    for frame in all_data.get(video).get('keypoints'):
        for person in frame.keys():
            persons_frames[person].append(frame.get(person)[:17])
            
    all_keypoints[video].append(persons_frames)


In [ ]:
# len(all_keypoints['62794_6'][0]['44'])

In [ ]:
#main_person_keypoints.get('80756_1').get('1860')[0]

In [ ]:
# if extract_main_person:
#     keypoints = main_person_keypoints.keys()
# else:
#     pass

In [ ]:
# pip install scikit-learn

In [ ]:
import random
from sklearn.model_selection import train_test_split

video_ids = list(all_keypoints.keys())
random.seed(42)

train_val, test = train_test_split(video_ids, test_size=0.10, random_state=42)

val_size = 0.1111
train, val = train_test_split(train_val, test_size=val_size, random_state=42)

split = {
    'train': train,
    'val': val,
    'test': test
}

In [ ]:
print(len(train), len(test), len(val))

In [ ]:
import numpy as np

coords = {}
confidences = {}

# Iteramos sobre todos los videos en all_keypoints
for video_id in all_keypoints.keys():
    persons_data = all_keypoints[video_id][0]  # dict: person_id -> list of frames

    person_ids = list(persons_data.keys())
    num_persons = len(person_ids)
    num_frames = max(len(persons_data[pid]) for pid in person_ids)
    num_keypoints = len(persons_data[person_ids[0]][0])  # assumed 17 keypoints

    # Inicializamos arrays vacíos
    keypoint_array = np.zeros((num_persons, num_frames, num_keypoints, 2), dtype='float32')
    score_array = np.zeros((num_persons, num_frames, num_keypoints), dtype='float32')


    for m, pid in enumerate(person_ids):
        frames = persons_data[pid]
        for t, frame in enumerate(frames):
            for v, kp in enumerate(frame):
                keypoint_array[m, t, v] = kp[:2]
                score_array[m, t, v] = kp[2] if len(kp) > 2 else 0.0

    coords[video_id] = keypoint_array
    confidences[video_id] = score_array


In [ ]:
final_dict = {}
final_dict['split'] = {'train': train, 'test': test, 'val': val}
final_dict['annotations'] = []

for video_id in all_keypoints.keys():
    final_dict['annotations'].append({
        'frame_dir': video_id,
        'total_frames': video_properties[video_id]['frame_count'],
        'img_shape': (video_properties[video_id]['height'], video_properties[video_id]['width']),
        'original_shape': (video_properties[video_id]['height'], video_properties[video_id]['width']),
        'label': 0,
        'keypoint': coords[video_id],  # shape [M, T, V, C]
        'keypoint_score': confidences[video_id]  # shape [M, T, V]
    })


In [ ]:
final_dict['annotations'][0]

In [ ]:
import pickle


with open(os.path.join(f'{sample_class}.pkl'), 'wb') as handle:
    pickle.dump(final_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)


In [ ]:
from joblib import load

obj = load("correct.pkl")
print(type(obj))

In [ ]:
obj.get('split')

In [ ]:
obj.get('annotations')